# Algorithmic Trading and Quantitative Strategies
## Part 11: Return Prediction and Portfolio Optimization
**Dr. Ayhan Yuksel, CFA, FDP, FRM, PRM**

Bogazici University, EC581

## Table of Contents

1. **Two-Step Framework: Return Prediction → Portfolio Construction**
2. **Return Prediction**
   - 2.1 Setup: Rebalance Frequency and Targets
   - 2.2 Regression vs. Classification
   - 2.3 Common Methods
   - 2.4 Converting Sign Forecasts into Expected Returns
   - 2.5 Feature Engineering — The Most Important Part
3. **Markowitz Portfolio Optimization**
   - 3.1 The Mean-Variance Problem
   - 3.2 Closed-Form Maximum-Sharpe Weights
   - 3.3 Long-Only and Other Constraints
   - 3.4 Estimating Volatilities and Covariances
   - 3.5 Beyond Plain Markowitz — Pointers
4. **End-to-End Application**
   - 4.1 Universe and Data
   - 4.2 Three Momentum Features
   - 4.3 Forward-Return Targets
   - 4.4 One-Shot Prediction at the Last Date
   - 4.5 Volatility Forecasts: Historical vs. EWMA
   - 4.6 EWMA Covariance Matrix
   - 4.7 Long-Only Mean-Variance Optimization
   - 4.8 Comparing the Three Approaches
5. **Discussion and Limitations**
6. **Exercises**

## 1. Two-Step Framework: Return Prediction → Portfolio Construction

Most quantitative equity strategies share a common backbone:

1. **Predict expected returns** for each asset over the next $H$ trading days using observable features (momentum, value, quality, macro, ...).
2. **Translate predictions into portfolio weights** through a portfolio optimizer that balances expected return against risk and constraints.

Both steps are *equally* important. A great signal coupled with a bad allocator (e.g., putting 100% in the noisiest top-pick) can ruin performance, and a great allocator cannot rescue a useless signal.

### Rebalance Frequency $H$

The first design choice is the **rebalance horizon** $H$ — how often we update the portfolio:

- $H = 1$ day: high-frequency / intraday
- $H = 5$ days (weekly): typical for medium-frequency systematic strategies
- $H = 21$ days (monthly): classic factor investing (NB_08)
- $H = 63$ days (quarterly) or longer: lower turnover, fundamentals-driven

Lower $H$ means more trades, higher transaction costs, but faster reaction to new information. Throughout this lecture we use $H = 5$ days.

### Where this notebook fits

- **NB_08** built cross-sectional factor signals; we now turn them into return forecasts and into a portfolio.
- **NB_06** introduced money-management rules; here we replace them with an optimizer.
- **NB_12** (next) tests the robustness of strategies built this way.

## 2. Return Prediction

### 2.1 Setup: Rebalance Frequency and Targets

Let $P_t$ be the price of an asset at time $t$ and define the **forward $H$-period log return**:

$$r_{t \to t+H} \;=\; \ln\!\left(\frac{P_{t+H}}{P_t}\right).$$

We want to forecast $r_{t \to t+H}$ from a feature vector $x_t$ that uses **only information observable up to time $t$**.

Two natural learning targets:

| Problem | Target | Output of the model |
|:---|:---|:---|
| **Regression** | $y_t = r_{t \to t+H}$ | A continuous estimate $\hat{\mu}_t$ |
| **Classification** | $y_t = \text{sign}(r_{t \to t+H}) \in \{-1, +1\}$ | A probability $\hat{p}_t = \Pr(r_{t \to t+H} > 0 \mid x_t)$ |

### 2.2 Regression vs. Classification

Regression directly predicts the **magnitude** of returns and feeds straight into a Markowitz optimizer. Its drawback: financial returns are extremely noisy, so a regression model spends most of its effort fitting noise. Out-of-sample $R^2$ values around **1–2%** are considered very good; values above $5\%$ are usually a sign of look-ahead bias.

Classification asks an easier question — *will the return be positive?* — and is more robust to outliers and non-Gaussian noise. The cost: the model loses information about *how much* the return will move, which a portfolio optimizer needs to size positions. We will recover the magnitude with a simple post-processing step in 2.4.

Rules of thumb:

- Use **regression** when the signal-to-noise ratio is reasonable, when relationships look approximately linear / smooth, and when magnitude information truly matters for sizing.
- Use **classification** when the data are very noisy, when you mainly care about direction (e.g., long-short signal generation), or when the relationship between features and returns is highly non-linear / non-monotonic at the tails.

### 2.3 Common Methods

All of these can be plugged into either a regression or a classification target.

| Method | Linear? | Handles interactions? | Interpretability | Overfitting risk |
|:---|:---:|:---:|:---:|:---:|
| **OLS / Logistic regression** | yes | only via manual interaction terms | high | low |
| **Ridge / Lasso / Elastic Net** | yes | manual | high | low |
| **Random Forest** | no | yes | medium | medium |
| **Gradient Boosting (XGBoost, LightGBM)** | no | yes | medium | medium-high |
| **Neural networks** | no | yes | low | high |

For an introductory pipeline we will use:

- **OLS** for the regression target.
- **Logistic regression** for the classification target, combined with the sign-to-return conversion in 2.4.
- **Random Forest regressor** as a simple non-linear alternative.

**XGBoost** and **LightGBM** are the workhorses of modern systematic shops, but they require careful regularization (depth, learning rate, early stopping) and add a dependency. We mention them here for completeness; the same pipeline applies.

### 2.4 Converting Sign Forecasts into Expected Returns

A classifier returns $\hat{p}_t = \Pr(r_{t \to t+H} > 0 \mid x_t)$. To plug this into a Markowitz optimizer we need a number with the same units as a return. Two simple recipes:

**(a) Soft conversion using historical conditional means.** Estimate from history

$$\mu^{+} \;=\; \mathbb{E}\!\left[\,r \mid r > 0\,\right], \qquad \mu^{-} \;=\; \mathbb{E}\!\left[\,r \mid r < 0\,\right],$$

then forecast

$$\hat{\mu}_t \;=\; \hat{p}_t \, \mu^{+} \;+\; (1 - \hat{p}_t) \, \mu^{-}.$$

**(b) Hard sign times average magnitude.** Round $\hat{p}_t$ to a hard $\pm 1$ vote and multiply by $|\bar r|$:

$$\hat{\mu}_t \;=\; \widehat{\text{sign}}_t \cdot \bar{|r|}.$$

Recipe (a) is smoother and is what we will use in the application. Crucially, **$\mu^{+}$ and $\mu^{-}$ must be estimated only from in-sample data** to avoid look-ahead bias.

### 2.5 Feature Engineering — The Most Important Part

Across academia and industry, feature design dominates model choice for return prediction. Spending a day choosing between OLS and Random Forest is much less productive than spending a day designing a clean, well-motivated feature.

**Common feature families:**

| Family | Examples |
|:---|:---|
| **Price / momentum** | 1-week, 1-month, 12–1 month returns; distance from moving average; RSI; trend strength (slope of regression on time) |
| **Volatility / risk** | rolling standard deviation, EWMA volatility, downside semi-deviation, max drawdown over a window |
| **Cross-sectional rank** | percentile rank of any of the above within the universe at time $t$ |
| **Valuation / fundamentals** | trailing P/E, P/B, dividend yield, EV/EBITDA, ROE (link to NB_08) |
| **Volume / liquidity** | dollar volume, Amihud illiquidity, bid-ask spread |
| **Macro** | yield curve slope, credit spread (HYG − IEF), USD index, oil, VIX |

**Practical rules:**

- **No look-ahead.** Every feature at time $t$ must use only data observable at $t$.
- **Stationarity.** Prefer returns and ratios over raw prices; standardize features cross-sectionally if you mix tickers.
- **Robustness.** Winsorize extreme percentiles (e.g., 1% / 99%) before modeling.
- **Parsimony.** A handful of well-designed features usually outperforms dozens of correlated ones.

## 3. Markowitz Portfolio Optimization

### 3.1 The Mean-Variance Problem

Once we have a vector of expected returns $\mu \in \mathbb{R}^N$ and a covariance matrix $\Sigma \in \mathbb{R}^{N \times N}$ for $N$ assets, Markowitz (1952) proposes to choose weights $w \in \mathbb{R}^N$ that solve

$$\max_{w} \;\; w^\top \mu \;-\; \frac{\gamma}{2}\, w^\top \Sigma w, \qquad \text{s.t. } \mathbf{1}^\top w = 1,$$

where $\gamma > 0$ is the risk-aversion parameter. Equivalently, sweeping $\gamma$ traces out the **efficient frontier** of portfolios with maximum return for a given level of risk.

### 3.2 Closed-Form Maximum-Sharpe Weights

Among all unconstrained fully-invested portfolios, the one that maximizes the Sharpe ratio

$$\text{SR}(w) \;=\; \frac{w^\top \mu}{\sqrt{w^\top \Sigma w}}$$

(here $\mu$ is interpreted as expected *excess* return) admits a closed-form solution:

$$\boxed{\;w^{*} \;\propto\; \Sigma^{-1} \mu\;}$$

Imposing the budget constraint $\mathbf{1}^\top w = 1$ gives the **tangency portfolio**:

$$w_{\text{tan}} \;=\; \frac{\Sigma^{-1} \mu}{\mathbf{1}^\top \Sigma^{-1} \mu}.$$

**Sketch of the derivation.** $\text{SR}(w)$ is scale-invariant in $w$, so we can fix $w^\top \Sigma w = 1$ and maximize $w^\top \mu$ via a Lagrangian. The first-order condition gives $\mu = 2\lambda \Sigma w$, hence $w \propto \Sigma^{-1} \mu$.

**Read this geometrically.** The optimizer (i) tilts the portfolio toward assets with high expected return ($\Sigma^{-1} \mu$ has the same sign pattern as $\mu$ when assets are roughly uncorrelated), and (ii) penalizes assets that are highly correlated with other holdings (because $\Sigma^{-1}$ down-weights collinear directions).

### 3.3 Long-Only and Other Constraints

In practice we almost always face additional constraints:

- **Long-only:** $w_i \ge 0$ for all $i$.
- **Position caps:** $w_i \le w_{\max}$ to avoid concentration.
- **Sector or factor exposure caps:** $\sum_{i \in S} w_i \le c_S$.
- **Turnover:** $\sum_i |w_i - w_i^{\text{prev}}| \le \tau$.

With *only* equality constraints there is still a closed-form solution, but with **inequality constraints** (long-only, caps, ...) the problem becomes a quadratic program and we solve it numerically. We use `scipy.optimize.minimize` with the SLSQP solver to maximize the Sharpe ratio under

$$w_i \ge 0, \qquad \sum_i w_i = 1.$$

The unconstrained closed-form is the right *intuition pump*; the long-only solver is the right *production tool*.

### 3.4 Estimating Volatilities and Covariances

The covariance matrix $\Sigma$ must be estimated from data. Two simple, widely-used approaches:

**Historical (rolling) standard deviation.** Over a window of $L$ days,

$$\hat{\sigma}^2_t \;=\; \frac{1}{L-1} \sum_{s=t-L+1}^{t} \big(r_s - \bar r\big)^2.$$

Pros: simple, unbiased. Cons: a sudden volatility regime change takes $\approx L$ days to appear in the estimate.

**Exponentially-weighted moving average (EWMA).**

$$\hat{\sigma}^2_t \;=\; \lambda \, \hat{\sigma}^2_{t-1} \;+\; (1-\lambda)\, r_t^2,$$

with $\lambda \in (0,1)$. RiskMetrics popularized $\lambda = 0.94$ for daily data, which corresponds to a half-life of $\ln(0.5)/\ln(0.94) \approx 11$ days. EWMA is more responsive to regime shifts and is the de-facto standard for short-horizon volatility forecasts.

The same recursion extends to **EWMA covariance**:

$$\hat{\Sigma}_t \;=\; \lambda \, \hat{\Sigma}_{t-1} \;+\; (1-\lambda)\, r_t r_t^\top.$$

### 3.5 Beyond Plain Markowitz — Pointers

Empirically, plain Markowitz is *very* sensitive to the inputs: small changes in $\hat{\mu}$ produce wildly different weights (the famous **error-maximization problem**). Practical fixes include:

- **Shrinkage** of the covariance matrix (Ledoit-Wolf, 2004) toward a structured target.
- **Black-Litterman** (1992): blend an equilibrium prior with investor views to stabilize $\hat{\mu}$.
- **Robust optimization**: optimize the worst case over a confidence set for $\hat{\mu}$.
- **Risk parity / equal risk contribution**: bypass $\hat{\mu}$ entirely and use $\hat{\Sigma}$ alone.

We will see in Section 5 why these matter.

## 4. End-to-End Application

We now put the pieces together on a small universe of US stocks:

1. Download 5 years of daily prices for 10 large-cap tickers.
2. Build three momentum features per ticker: short-term, medium-term, long-term.
3. Define the target as the 5-day forward log return (and its sign).
4. **At the most recent date**, predict expected returns three ways:
   - **OLS regression** on the three features.
   - **Logistic regression** on the sign target, converted to a return via $\mu^{+}, \mu^{-}$.
   - **Random Forest regression** as a simple non-linear alternative.
5. Forecast volatilities (historical vs. EWMA) and the EWMA covariance matrix.
6. Solve a long-only Markowitz problem for each of the three $\hat{\mu}$ vectors and compare.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

RNG_SEED = 42
np.random.seed(RNG_SEED)

### 4.1 Universe and Data

Ten large-cap US stocks across four broad sectors. Five years of daily prices is enough to fit simple per-ticker models and have sensible covariance estimates.

In [ ]:
tickers = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'NVDA',
           'JPM', 'XOM', 'PG', 'UNH', 'KO']

end_date = pd.Timestamp.today().strftime('%Y-%m-%d')
start_date = (pd.Timestamp.today() - pd.DateOffset(years=5)).strftime('%Y-%m-%d')

raw = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True, progress=False)
prices = raw['Close'].dropna(how='all')
prices = prices[tickers].dropna()

log_ret = np.log(prices / prices.shift(1)).dropna()

print(f'Date range : {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'Daily bars : {len(prices)}')
print(f'Universe   : {len(tickers)} tickers')
prices.tail()

In [ ]:
norm = prices / prices.iloc[0]
norm.plot(figsize=(14, 6), linewidth=1.1)
plt.title('Cumulative Total-Return Index (Base = 1.0)', fontweight='bold')
plt.ylabel('Indexed Price')
plt.grid(True, alpha=0.3)
plt.legend(ncol=5, loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

### 4.2 Three Momentum Features

We build three features per ticker, all expressed as past log returns:

| Feature | Definition | Intuition |
|:---|:---|:---|
| `mom_short` | 5-day return | Short-term continuation / reversal |
| `mom_med`   | 21-day (1-month) return | Medium-term trend |
| `mom_long`  | 252-day return skipping the last 21 days ("12−1 month") | Classical Jegadeesh-Titman momentum |

Each feature uses only *past* prices and is therefore safe to use at time $t$.

In [ ]:
log_p = np.log(prices)

mom_short = log_p - log_p.shift(5)
mom_med   = log_p - log_p.shift(21)
mom_long  = log_p.shift(21) - log_p.shift(252)

features = pd.concat(
    {'mom_short': mom_short, 'mom_med': mom_med, 'mom_long': mom_long},
    axis=1
)

features = features.dropna()
print(f'Feature panel shape: {features.shape}')
features.tail().round(4)

### 4.3 Forward-Return Targets

Pick a rebalance horizon $H = 5$ days. For each ticker we form

$$y_t = \ln\!\left(\frac{P_{t+H}}{P_t}\right), \qquad \tilde y_t = \text{sign}(y_t).$$

The last $H$ rows are dropped from the training set because their target is unobservable, but the **feature row at the very last available date is still usable for prediction**.

In [ ]:
H = 5

fwd_ret = np.log(prices.shift(-H) / prices)
fwd_sign = np.sign(fwd_ret)

print(f'Forward {H}-day return summary across tickers:')
fwd_ret.describe().round(4)

### 4.4 One-Shot Prediction at the Last Date

We fit one model **per ticker** on all rows where both features and target are observed, then use that model to predict $\hat{\mu}$ for the **most recent feature vector**, whose target is in the future and unknown.

Per-ticker is the simplest baseline: it gives each stock its own momentum-loading. A pooled cross-sectional model (one regression across all tickers, possibly with ticker fixed effects or a panel structure) is the natural next step.

In [ ]:
feat_names = ['mom_short', 'mom_med', 'mom_long']

def per_ticker_dataset(ticker):
    """Return (X_train, y_train, x_last) aligned and cleaned for one ticker."""
    X = pd.concat({f: features[f][ticker] for f in feat_names}, axis=1)
    y = fwd_ret[ticker]
    df = pd.concat([X, y.rename('y')], axis=1)
    train = df.dropna()
    x_last = X.iloc[[-1]].dropna()
    return train[feat_names], train['y'], x_last

sample_X, sample_y, sample_last = per_ticker_dataset(tickers[0])
print(f'Example for {tickers[0]}: train={sample_X.shape}, last_x={sample_last.shape}')
sample_last.round(4)

In [ ]:
mu_ols = pd.Series(index=tickers, dtype=float)
ols_coefs = {}

for tic in tickers:
    X, y, x_last = per_ticker_dataset(tic)
    Xc = sm.add_constant(X)
    model = sm.OLS(y, Xc).fit()
    ols_coefs[tic] = model.params
    mu_ols[tic] = float(model.predict(sm.add_constant(x_last, has_constant='add')).iloc[0])

ols_coef_df = pd.DataFrame(ols_coefs).T
print('Per-ticker OLS coefficients (intercept + 3 features):')
ols_coef_df.round(4)

In [ ]:
mu_logit = pd.Series(index=tickers, dtype=float)
p_up = pd.Series(index=tickers, dtype=float)
mu_plus = pd.Series(index=tickers, dtype=float)
mu_minus = pd.Series(index=tickers, dtype=float)

for tic in tickers:
    X, y, x_last = per_ticker_dataset(tic)
    sign_y = (y > 0).astype(int)

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X.values, sign_y.values)
    p = float(clf.predict_proba(x_last.values)[0, 1])

    mp = y[y > 0].mean()
    mm = y[y < 0].mean()

    p_up[tic] = p
    mu_plus[tic] = mp
    mu_minus[tic] = mm
    mu_logit[tic] = p * mp + (1 - p) * mm

logit_table = pd.DataFrame({
    'P(up)': p_up,
    'mu+': mu_plus,
    'mu-': mu_minus,
    'mu_hat (logit)': mu_logit,
})
logit_table.round(4)

In [ ]:
mu_rf = pd.Series(index=tickers, dtype=float)

for tic in tickers:
    X, y, x_last = per_ticker_dataset(tic)
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=20,
        random_state=RNG_SEED,
        n_jobs=1,
    )
    rf.fit(X.values, y.values)
    mu_rf[tic] = float(rf.predict(x_last.values)[0])

mu_compare = pd.concat(
    {'OLS': mu_ols, 'Logistic+SignToReturn': mu_logit, 'RandomForest': mu_rf},
    axis=1,
)
print(f'Forecasted {H}-day log returns at the last available date:')
mu_compare.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(tickers))
width = 0.27
ax.bar(x - width, mu_ols.values * 100, width, label='OLS')
ax.bar(x,         mu_logit.values * 100, width, label='Logistic+SignToReturn')
ax.bar(x + width, mu_rf.values * 100,   width, label='Random Forest')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(tickers)
ax.set_ylabel(f'Predicted {H}-day log return (%)')
ax.set_title('Return Forecasts at the Last Date — Three Models', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

**Reading the chart.** The three approaches frequently agree on the *sign* of the forecast but disagree on *magnitude*. The OLS forecasts are the smoothest because they extrapolate linearly from the latest momentum reading. The logistic-based forecasts are bounded by $|\mu^{+}|$ and $|\mu^{-}|$ — they cannot exceed the historical average of positive (or negative) outcomes — so they are typically the most conservative. The Random Forest can produce sharper forecasts when the latest features fall into a well-populated leaf with a clear directional bias.

### 4.5 Volatility Forecasts: Historical vs. EWMA

We compute two volatility estimates per ticker and compare them visually for two representative names. The EWMA volatility uses $\lambda = 0.94$ (RiskMetrics).

In [ ]:
LAMBDA = 0.94
ROLL_WINDOW = 60

hist_vol_daily = log_ret.rolling(ROLL_WINDOW).std()

ewma_var_daily = log_ret.pow(2).ewm(alpha=1 - LAMBDA, adjust=False).mean()
ewma_vol_daily = np.sqrt(ewma_var_daily)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, tic in zip(axes, ['NVDA', 'KO']):
    ax.plot(hist_vol_daily.index, hist_vol_daily[tic] * np.sqrt(252) * 100,
            label=f'Rolling {ROLL_WINDOW}d', linewidth=1.2)
    ax.plot(ewma_vol_daily.index, ewma_vol_daily[tic] * np.sqrt(252) * 100,
            label=f'EWMA λ={LAMBDA}', linewidth=1.2)
    ax.set_title(f'{tic}: Annualized Volatility (%)', fontweight='bold')
    ax.set_ylabel('Vol (% ann.)')
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

EWMA tracks regime shifts — spikes in 2022 and recent moves — noticeably faster than a 60-day rolling window. For tactical allocators that rebalance every few days, EWMA is usually the better choice.

### 4.6 EWMA Covariance Matrix

We apply the same recursion to the full $N \times N$ covariance matrix, then convert it to the **5-day** scale to match the horizon of $\hat{\mu}$:

$$\hat{\Sigma}^{(H)} \;\approx\; H \cdot \hat{\Sigma}^{(\text{daily})}.$$

In [ ]:
def ewma_cov(returns: pd.DataFrame, lam: float = 0.94) -> np.ndarray:
    """EWMA covariance recursion. Returns the matrix at the last date."""
    R = returns.values
    n = R.shape[1]
    cov = np.cov(R[:60].T)
    for t in range(60, len(R)):
        r = R[t].reshape(-1, 1)
        cov = lam * cov + (1 - lam) * (r @ r.T)
    return cov

Sigma_daily = ewma_cov(log_ret, LAMBDA)
Sigma_H = H * Sigma_daily

Sigma_H_df = pd.DataFrame(Sigma_H, index=tickers, columns=tickers)
vol_H = pd.Series(np.sqrt(np.diag(Sigma_H)), index=tickers, name=f'sigma_{H}d')

D_inv = np.diag(1.0 / np.sqrt(np.diag(Sigma_H)))
Corr = D_inv @ Sigma_H @ D_inv
Corr_df = pd.DataFrame(Corr, index=tickers, columns=tickers)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(Corr_df.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(tickers)))
ax.set_yticks(range(len(tickers)))
ax.set_xticklabels(tickers, rotation=45, ha='right')
ax.set_yticklabels(tickers)
for i in range(len(tickers)):
    for j in range(len(tickers)):
        ax.text(j, i, f'{Corr_df.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title(f'EWMA Correlation Matrix at Last Date (λ={LAMBDA})', fontweight='bold')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print(f'\n{H}-day forecast volatilities (%):')
(vol_H * 100).round(2)

### 4.7 Long-Only Mean-Variance Optimization

We solve

$$\max_{w} \;\; \frac{w^\top \hat{\mu}}{\sqrt{w^\top \hat{\Sigma}^{(H)} w}}, \qquad \text{s.t. } w_i \ge 0,\;\; \sum_i w_i = 1,$$

for each of the three $\hat{\mu}$ vectors. The covariance matrix $\hat{\Sigma}^{(H)}$ is the same in all three runs — only the expected-return input changes.

In [ ]:
def long_only_max_sharpe(mu: np.ndarray, Sigma: np.ndarray) -> np.ndarray:
    """Maximize w' mu / sqrt(w' Sigma w) subject to w_i >= 0, sum w_i = 1."""
    n = len(mu)
    w0 = np.full(n, 1.0 / n)

    def neg_sharpe(w):
        ret = float(w @ mu)
        vol = float(np.sqrt(w @ Sigma @ w))
        return -ret / vol if vol > 0 else 0.0

    cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},)
    bnds = tuple((0.0, 1.0) for _ in range(n))

    res = minimize(neg_sharpe, w0, method='SLSQP', bounds=bnds, constraints=cons,
                   options={'maxiter': 500, 'ftol': 1e-10})
    w = np.clip(res.x, 0.0, None)
    if w.sum() == 0:
        return w0
    return w / w.sum()

weights = {}
for label, mu_series in [('OLS', mu_ols),
                          ('Logistic+SignToReturn', mu_logit),
                          ('RandomForest', mu_rf)]:
    w = long_only_max_sharpe(mu_series.values, Sigma_H)
    weights[label] = pd.Series(w, index=tickers)

weights_df = pd.DataFrame(weights)
print('Long-only max-Sharpe weights:')
(weights_df * 100).round(2)

### 4.8 Comparing the Three Approaches

We tabulate (i) the resulting weights and (ii) the optimizer's *predicted* portfolio statistics: expected $H$-day return, $H$-day volatility, and Sharpe ratio under the assumed $\hat{\mu}$ and $\hat{\Sigma}^{(H)}$.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(tickers))
width = 0.27
ax.bar(x - width, weights['OLS'].values * 100, width, label='OLS')
ax.bar(x,         weights['Logistic+SignToReturn'].values * 100, width, label='Logistic+SignToReturn')
ax.bar(x + width, weights['RandomForest'].values * 100, width, label='Random Forest')
ax.set_xticks(x)
ax.set_xticklabels(tickers)
ax.set_ylabel('Portfolio weight (%)')
ax.set_title('Long-Only Max-Sharpe Weights — Three Approaches', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for label, mu_series in [('OLS', mu_ols),
                          ('Logistic+SignToReturn', mu_logit),
                          ('RandomForest', mu_rf)]:
    w = weights[label].values
    mu = mu_series.values
    port_ret = float(w @ mu)
    port_vol = float(np.sqrt(w @ Sigma_H @ w))
    sharpe_H = port_ret / port_vol if port_vol > 0 else np.nan
    rows.append({
        'Approach': label,
        f'E[r_{H}d] (%)': port_ret * 100,
        f'sigma_{H}d (%)': port_vol * 100,
        f'Sharpe ({H}d)': sharpe_H,
        'Sharpe (ann.)': sharpe_H * np.sqrt(252 / H),
        'Top weight': f"{weights[label].idxmax()} ({weights[label].max()*100:.1f}%)",
        'N held (>1%)': int((weights[label] > 0.01).sum()),
    })

summary = pd.DataFrame(rows).set_index('Approach')
summary.round(3)

**What to read in the table.**

- The three approaches share the same covariance input, so all differences come from $\hat{\mu}$.
- The OLS portfolio is usually the most concentrated: small differences in $\hat{\mu}$ across stocks get amplified through $\Sigma^{-1}$ (this is the error-maximization phenomenon).
- The logistic-based portfolio is more diversified because $\hat{\mu}$ is bounded between $\mu^{+}$ and $\mu^{-}$ — the gaps between assets are smaller.
- Random Forest sits between the two and tends to pick a few stocks on which the trees agree strongly.

Important caveat: the Sharpe ratios in the table are **predicted** under the model's own beliefs. They are not out-of-sample performance. The whole point of NB_12 is to test whether such forecasts hold up in reality.

## 5. Discussion and Limitations

### Signal-to-noise is brutally low

Daily and weekly equity returns are dominated by noise. Even a *very* good cross-sectional return-prediction model has out-of-sample $R^2$ around 1–2%. Expect headline numbers in our example to be small in absolute value, with relatively wide confidence intervals.

### Markowitz amplifies estimation error

Suppose two assets have *true* expected returns 1% and 2% and we estimate them at 0.8% and 2.5%. The unconstrained optimizer treats the second asset as roughly three times more attractive than the first, even though their true Sharpe ratios are nearly identical. With long-only constraints the effect is muted but still present — you saw it in 4.7 where small differences in $\hat{\mu}$ produced very different concentrations.

This is why in many realistic studies:

- **Equal-weight** beats Markowitz out-of-sample (DeMiguel, Garlappi & Uppal, 2009).
- **Risk-parity** matches Markowitz with much smaller turnover.
- **Black-Litterman** and **shrinkage** (Ledoit-Wolf) are popular fixes precisely because they pull $\hat{\mu}$ and $\hat{\Sigma}$ toward stable priors.

### Where to go next

- Replace the per-ticker model with a **pooled cross-sectional** regression. Standardize features cross-sectionally each date.
- Add **fundamental features** (P/E, ROE) and **macro features**.
- Move from one-shot prediction to a **walk-forward backtest** with monthly rebalancing and transaction costs (NB_06 style).
- Apply **shrinkage** (`sklearn.covariance.LedoitWolf`) to $\hat{\Sigma}$ and compare weight stability.
- Stress-test the resulting strategy with the techniques in NB_12.

## 6. Exercises

**E1.** Replace the EWMA covariance with a plain rolling 60-day sample covariance and re-run the long-only optimization for the OLS forecast. Report the new weights and discuss whether they look more or less stable than the EWMA version.

**E2.** Add one fundamental feature — trailing P/E from `yfinance` (Ticker.info or financial statements) — to the three momentum features. Refit the OLS regression and report (a) the t-statistic on the new feature for two tickers and (b) how the resulting weight vector compares to the momentum-only version. Discuss any sign or magnitude changes you observe.